In [2]:
import sys
import os
import requests
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

sys.path.append("..")
from data.base import get_connection

print("Dependencies loaded successfully.")

# 1. Ensure external directory exists
os.makedirs('../data/external', exist_ok=True)
weather_csv = '../data/external/nyc_weather_daily.csv'

# 2. Automatically download daily historical weather for NYC Central Park if not cached
if not os.path.exists(weather_csv):
    print("Fetching daily NYC weather data (2016-2026)...")
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": 40.7829,       # NYC Central Park Station
        "longitude": -73.9654,
        "start_date": "2016-01-01",
        "end_date": "2026-06-30",
        "daily": ["precipitation_sum", "snowfall_sum", "temperature_2m_mean", "temperature_2m_min"],
        "timezone": "America/New_York"
    }
    resp = requests.get(url, params=params)
    resp.raise_for_status()
    data = resp.json()['daily']
    
    weather_df = pd.DataFrame({
        'date': pd.to_datetime(data['time']),
        'prcp_mm': data['precipitation_sum'],
        'snowfall_cm': data['snowfall_sum'],
        'tmin_c': data['temperature_2m_min']
    })
    # Convert snowfall cm to inches: 1 cm ~ 0.3937 inches
    weather_df['snowfall_inches'] = weather_df['snowfall_cm'] * 0.3937
    weather_df['is_freezing'] = (weather_df['tmin_c'] <= 0).astype(int)
    weather_df.to_csv(weather_csv, index=False)
    print(f"Weather data saved to {weather_csv}")
else:
    print(f"Loading cached weather data from {weather_csv}")
    weather_df = pd.read_csv(weather_csv)
    weather_df['date'] = pd.to_datetime(weather_df['date'])

weather_df.head()

Dependencies loaded successfully.
Loading cached weather data from ../data/external/nyc_weather_daily.csv


,date,prcp_mm,snowfall_cm,tmin_c,snowfall_inches,is_freezing
0,2016-01-01,0.0,0.0,0.1,0.0,0
1,2016-01-02,0.0,0.0,-1.8,0.0,1
2,2016-01-03,0.0,0.0,-1.3,0.0,1
3,2016-01-04,0.0,0.0,-8.7,0.0,1
4,2016-01-05,0.0,0.0,-10.3,0.0,1


In [3]:
con = get_connection()

query = """
    WITH cleaned_tickets AS (
        SELECT 
            CAST(t.issue_date AS DATE) AS date,
            CAST(t.violation_code AS INTEGER) AS violation_code,
            REGEXP_REPLACE(
                REGEXP_REPLACE(
                    REGEXP_REPLACE(
                        REGEXP_REPLACE(
                            TRIM(UPPER(COALESCE(t.street_name, '') || ' ' || COALESCE(t.intersecting_street, ''))),
                            '\\b(NB|SB|EB|WB|N/B|S/B|E/B|W/B)\\b', '', 'g'
                        ),
                        '\\(.*?\\)', '', 'g'
                    ),
                    '@', ' AND ', 'g'
                ),
                '\\s+', ' ', 'g'
            ) || ', NEW YORK, NY' AS search_query
        FROM tickets_all t
        WHERE CAST(t.violation_code AS INTEGER) IN (7, 36)
          AND t.issue_date >= '2016-01-01'
          AND t.street_name IS NOT NULL 
          AND t.intersecting_street IS NOT NULL
    ),
    daily_tickets AS (
        SELECT 
            t.date,
            t.search_query,
            p.camera_type,
            COUNT(*) AS tickets_issued
        FROM cleaned_tickets t
        INNER JOIN read_csv_auto('../data/current_active_permanent_cameras_2026.csv') p 
            ON t.search_query = p.search_query
        GROUP BY 1, 2, 3
    ),
    daily_crashes AS (
        SELECT 
            CAST(crash_date AS DATE) AS date,
            search_query,
            COUNT(*) AS crash_count,
            SUM(COALESCE(number_of_persons_injured, 0)) AS injured_count
        FROM camera_crash_buffers
        GROUP BY 1, 2
    )
    SELECT 
        COALESCE(t.date, c.date) AS date,
        COALESCE(t.search_query, c.search_query) AS search_query,
        COALESCE(t.camera_type, 'Speed') AS camera_type,
        COALESCE(t.tickets_issued, 0) AS tickets_issued,
        COALESCE(c.crash_count, 0) AS crash_count,
        COALESCE(c.injured_count, 0) AS injured_count
    FROM daily_tickets t
    FULL OUTER JOIN daily_crashes c 
      ON t.date = c.date AND t.search_query = c.search_query
    ORDER BY date, search_query
"""

try:
    print("Executing daily extraction in DuckDB...")
    daily_panel = con.sql(query).df()
    print(f"Extracted {len(daily_panel):,} daily intersection-level records.")
finally:
    con.close()

daily_panel['date'] = pd.to_datetime(daily_panel['date'])
daily_panel.head()

Executing daily extraction in DuckDB...
Extracted 664,104 daily intersection-level records.


,date,search_query,camera_type,tickets_issued,crash_count,injured_count
0,2016-01-01,"10TH AVE AND LITTLE W 12TH ST, NEW YORK, NY",Speed,0,1,0.0
1,2016-01-01,"21ST ST AND 37TH AV E, NEW YORK, NY",Speed,0,1,0.0
2,2016-01-01,"77TH ST AND 30TH AV E, NEW YORK, NY",Speed,0,1,0.0
3,2016-01-01,"ATLANTIC AVE AND LE FFERTS BLVD, NEW YORK, NY",Speed,0,1,1.0
4,2016-01-01,"ATLANTIC AVE AND PENNSYLVANIA AVE, NEW YORK, NY",Speed,0,1,0.0


In [4]:
# Merge daily panel with weather records
panel = daily_panel.merge(weather_df, on='date', how='inner')

# 1. Define Snowstorm Shock Threshold (>= 2 inches of snowfall)
panel['is_snow_storm'] = (panel['snowfall_inches'] >= 2.0).astype(int)

# 2. Add Storm Lag Windows (Day +1 and Day +2 residual road conditions)
panel = panel.sort_values(['search_query', 'date']).reset_index(drop=True)
panel['storm_lag_1'] = panel.groupby('search_query')['is_snow_storm'].shift(1).fillna(0).astype(int)
panel['storm_lag_2'] = panel.groupby('search_query')['is_snow_storm'].shift(2).fillna(0).astype(int)

# 3. Temporal Controls
panel['month'] = panel['date'].dt.month.astype(str)
panel['year'] = panel['date'].dt.year.astype(str)
panel['day_of_week'] = panel['date'].dt.dayofweek.astype(str)

print("Weather panel successfully constructed:")
print(f" - Total Observations: {len(panel):,}")
print(f" - Identified Snow Storm Days: {weather_df['snowfall_inches'].ge(2.0).sum()}")
panel[['date', 'search_query', 'tickets_issued', 'crash_count', 'snowfall_inches', 'is_snow_storm', 'is_freezing']].sample(5)

Weather panel successfully constructed:
 - Total Observations: 660,456
 - Identified Snow Storm Days: 38


,date,search_query,tickets_issued,crash_count,snowfall_inches,is_snow_storm,is_freezing
84218,2026-03-10,"ATLANTIC AVE AND EA STERN PKWY, NEW YORK, NY",0,1,0.000000,0,0
250802,2022-12-12,"FARMERS BLVD AND LI NDEN BLVD, NEW YORK, NY",0,1,0.275590,0,1
85768,2021-09-24,"ATLANTIC AVE AND FT GREENE PL, NEW YORK, NY",7,0,0.000000,0,0
546492,2021-10-30,"ROCKAWAY PKWY AND SEAVIEW AVE, NEW YORK, NY",5,0,0.000000,0,0
466137,2025-12-13,"PENNSYLVANIA AVE AND LINDEN BLVD, NEW YORK, NY",28,0,0.248031,0,1


In [5]:
print("=== Model 1: Weather Impact on Camera Ticket Volume (Clustered OLS) ===")

panel['log_tickets'] = np.log1p(panel['tickets_issued'])

ticket_model = smf.ols(
    """
    log_tickets ~ is_snow_storm + storm_lag_1 + storm_lag_2 + is_freezing 
                  + C(camera_type) + C(month) + C(year) + C(day_of_week)
    """,
    data=panel
).fit(cov_type='cluster', cov_kwds={'groups': panel['search_query']})

print(ticket_model.summary().tables[1])

=== Model 1: Weather Impact on Camera Ticket Volume (Clustered OLS) ===
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                   1.7573      0.050     35.139      0.000       1.659       1.855
C(camera_type)[T.Speed]    -1.9223      0.077    -25.112      0.000      -2.072      -1.772
C(month)[T.10]              0.2001      0.011     18.552      0.000       0.179       0.221
C(month)[T.11]              0.1497      0.009     17.067      0.000       0.133       0.167
C(month)[T.12]              0.0886      0.007     13.238      0.000       0.075       0.102
C(month)[T.2]               0.0256      0.005      5.517      0.000       0.017       0.035
C(month)[T.3]               0.0975      0.007     13.980      0.000       0.084       0.111
C(month)[T.4]               0.1304      0.010     13.540      0.000       0.111       0.149
C(month)

In [6]:
print("=== Model 2: Weather Impact on Crashes at Camera Intersections (Poisson GLM) ===")

crash_model = smf.glm(
    """
    crash_count ~ is_snow_storm + storm_lag_1 + storm_lag_2 + is_freezing 
                  + C(camera_type) + C(month) + C(year) + C(day_of_week)
    """,
    data=panel,
    family=sm.families.Poisson()
).fit(cov_type='cluster', cov_kwds={'groups': panel['search_query']})

print(crash_model.summary().tables[1])

# Calculate empirical percentage changes: (exp(coef) - 1) * 100
snow_coef = crash_model.params['is_snow_storm']
lag1_coef = crash_model.params['storm_lag_1']

pct_change_crashes = (np.exp(snow_coef) - 1) * 100
pct_change_lag1 = (np.exp(lag1_coef) - 1) * 100

print(f"\nImmediate Snowstorm Crash Impact: {pct_change_crashes:+.2f}%")
print(f"Day-After Residual Storm Crash Impact: {pct_change_lag1:+.2f}%")

=== Model 2: Weather Impact on Crashes at Camera Intersections (Poisson GLM) ===
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                  -2.9395      0.082    -35.905      0.000      -3.100      -2.779
C(camera_type)[T.Speed]     2.9857      0.085     35.185      0.000       2.819       3.152
C(month)[T.10]              0.0065      0.007      0.888      0.375      -0.008       0.021
C(month)[T.11]              0.0036      0.005      0.658      0.510      -0.007       0.014
C(month)[T.12]              0.0034      0.006      0.573      0.567      -0.008       0.015
C(month)[T.2]               0.0075      0.005      1.467      0.142      -0.003       0.017
C(month)[T.3]              -0.0063      0.005     -1.236      0.217      -0.016       0.004
C(month)[T.4]              -0.0217      0.008     -2.765      0.006      -0.037      -0.006